In [ ]:
import pandas as pd

def update_cumulative_roas(df):
    # Ensure correct data types to avoid issues with NaN comparison
    df['incremental_net_roas'] = pd.to_numeric(df['incremental_net_roas'], errors='coerce')

    # Sort by 'game_id' and 'cohort' to ensure proper group ordering
    df = df.sort_values(by=['game_id', 'cohort'])

    # Group by 'game_id' and 'cohort'
    for (game_id, cohort), group in df.groupby(['game_id', 'cohort']):
        # Loop over the group to modify the cumulative columns continuously
        cumulative_bad_wa = cumulative_expected_wa = cumulative_good_wa = None
        for i in range(len(group)):  # Iterate over each row in the group
            current_row = group.iloc[i]
        
            if(i==0):      # For the first row, initialize cumulative values
                cumulative_bad_wa = current_row['cumulative_net_roas_bad_wa']
                cumulative_expected_wa = current_row['cumulative_net_roas_expected_wa']
                cumulative_good_wa = current_row['cumulative_net_roas_good_wa']
            
            if pd.isnull(current_row['incremental_net_roas']):  # Check if incremental_net_roas is NaN
                # Update the cumulative values
                cumulative_bad_wa += current_row['incremental_net_roas_bad_pred']
                cumulative_expected_wa += current_row['incremental_net_roas_expected_pred']
                cumulative_good_wa += current_row['incremental_net_roas_good_pred']
                
                # Update the DataFrame with the new cumulative values
                df.loc[current_row.name, 'cumulative_net_roas_bad_wa'] = cumulative_bad_wa
                df.loc[current_row.name, 'cumulative_net_roas_expected_wa'] = cumulative_expected_wa
                df.loc[current_row.name, 'cumulative_net_roas_good_wa'] = cumulative_good_wa
            else:
                # Reset the cumulative values if incremental_net_roas is not NaN
                cumulative_bad_wa = current_row['cumulative_net_roas_bad_wa']
                cumulative_expected_wa = current_row['cumulative_net_roas_expected_wa']
                cumulative_good_wa = current_row['cumulative_net_roas_good_wa']

            if pd.isnull(current_row['cumulative_net_roas_bad_pred']):
                df.loc[current_row.name, 'cumulative_net_roas_bad_pred'] = current_row['cumulative_net_roas_bad']
                df.loc[current_row.name, 'cumulative_net_roas_expected_pred'] = current_row['cumulative_net_roas_expected']
                df.loc[current_row.name, 'cumulative_net_roas_good_pred'] = current_row['cumulative_net_roas_good']

                df.loc[current_row.name, 'cumulative_net_roas_bad_wa'] = current_row['cumulative_net_roas_bad']
                df.loc[current_row.name, 'cumulative_net_roas_expected_wa'] = current_row['cumulative_net_roas_expected']
                df.loc[current_row.name, 'cumulative_net_roas_good_wa'] = current_row['cumulative_net_roas_good']


    return df


In [ ]:
import pandas as pd
from datetime import datetime, timezone

df1 = pd.read_csv('final_M1.csv')

df1 = update_cumulative_roas(df1)

# Get the current date and time
current_datetime = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S.%f') + ' UTC'

columns_to_update = ['updated_at', 'created_at']  
df1 [columns_to_update] = current_datetime

df1.to_csv('final_M1.csv',index=False)